# Step 1: Generate reasoning traces

This notebook gives questions from **GSM8K** (math) and **CommonsenseQA** (commonsense) to one small language model, saves its step-by-step answers ("traces"), and checks whether each final answer is correct.

**Before running, in the Kaggle notebook settings (right-hand panel):**
- **Accelerator:** GPU T4 ×2
- **Internet:** On (requires a phone-verified Kaggle account)

**How to use it:** change the settings in cell 1, then run all cells from top to bottom (*Run All*).

- **First run:** `MODEL = "qwen2.5-3b"`, `N_QUESTIONS = 10`. Read the outputs.
- **Later:** run again with the other two models, then with `N_QUESTIONS = 300`.

## 1. Settings

Only this cell changes between runs. Keep `SEED` the same for every model, so all three models answer **the same questions**.

In [ ]:
MODEL = "qwen2.5-3b"      # "qwen2.5-3b", "llama3.2-3b" or "phi4-mini"
N_QUESTIONS = 10          # questions per dataset: 10 for testing, 300 for the real run
SEED = 2026               # fixed random seed, so the question sample is always the same
MAX_NEW_TOKENS = 512      # maximum length of each answer
BATCH_SIZE = 8            # questions processed at once; lower it if you get an out-of-memory error

MODELS = {
    "qwen2.5-3b": "Qwen/Qwen2.5-3B-Instruct",
    "llama3.2-3b": "meta-llama/Llama-3.2-3B-Instruct",   # needs approved access on Hugging Face
    "phi4-mini": "microsoft/Phi-4-mini-instruct",
}

## 2. Setup

**2a.** Update `transformers` and `huggingface_hub` (Phi-4-mini and Llama 3.2 need a recent version).

- Run this cell **first**, before any other cell imports `transformers`.
- Red "dependency conflict" warnings from pip are normal on Kaggle and can be ignored.
- If a later cell complains about versions: **Run → Restart & clear cell outputs**, then **Run All**. A restart erases all variables, so always run from the top afterwards.

In [ ]:
!pip install --upgrade transformers huggingface_hub

**2b.** Load the libraries and log in to Hugging Face. The login is only needed for **Llama**:
- **Recommended:** add your token once in **Add-ons → Secrets** with the name `HF_TOKEN`; the login is then automatic.
- **Otherwise:** a box appears where you paste your token.

For Qwen and Phi the login is skipped.

In [ ]:
import json, random, re
import pandas as pd
import torch
import transformers
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

print("transformers", transformers.__version__, "| torch", torch.__version__)
print("GPUs:", torch.cuda.device_count())

from huggingface_hub import login

if MODEL.startswith("llama"):
    try:
        from kaggle_secrets import UserSecretsClient
        login(UserSecretsClient().get_secret("HF_TOKEN"))
        print("Logged in with the HF_TOKEN secret")
    except Exception:
        login()   # paste your Hugging Face token in the box that appears
else:
    print("No login needed for", MODEL)

**2c. Access check.** Downloads only the model's small configuration file, which takes a few seconds. If you get a *gated repo* / *403* error here, your Llama access hasn't been approved yet, or you aren't logged in.

In [ ]:
from transformers import AutoConfig

AutoConfig.from_pretrained(MODELS[MODEL])
print("Access OK:", MODELS[MODEL])

## 3. Load the questions

- **GSM8K:** `test` split (1,319 questions). The correct answer is the number after `####` in the solution.
- **CommonsenseQA:** `validation` split (1,221 questions, 5 options A–E). The test split has no public answers.

We draw `N_QUESTIONS` random questions from each, using `SEED`.

In [ ]:
gsm8k = load_dataset("openai/gsm8k", "main", split="test")
csqa = load_dataset("tau/commonsense_qa", split="validation")

def sample_indices(n_total, n, seed):
    return sorted(random.Random(seed).sample(range(n_total), n))

questions = []
for i in sample_indices(len(gsm8k), N_QUESTIONS, SEED):
    row = gsm8k[i]
    questions.append({
        "dataset": "gsm8k",
        "question_id": f"gsm8k-{i}",
        "question": row["question"],
        "options": None,
        "gold_answer": row["answer"].split("####")[-1].strip().replace(",", ""),
    })
for i in sample_indices(len(csqa), N_QUESTIONS, SEED):
    row = csqa[i]
    questions.append({
        "dataset": "csqa",
        "question_id": row["id"],
        "question": row["question"],
        "options": dict(zip(row["choices"]["label"], row["choices"]["text"])),
        "gold_answer": row["answerKey"],
    })

print(len(questions), "questions loaded")
questions[0]

## 4. The prompts

Zero-shot (no worked examples), the same for every model. They ask for numbered steps, so each step has a number for "first wrong step", and a fixed `Final answer:` line, so the answer can be checked automatically. The reasons are in the plan, section 5.

In [ ]:
MATH_PROMPT = """Solve the following math problem. Reason step by step.
Write each step on its own line, numbered as "Step 1:", "Step 2:", and so on.
After the last step, write the final answer on its own line in exactly this format:
Final answer: <number>

Problem: {question}"""

CSQA_PROMPT = """Answer the following multiple-choice question. Reason step by step.
Write each step on its own line, numbered as "Step 1:", "Step 2:", and so on.
After the last step, write the letter of your chosen option on its own line in exactly this format:
Final answer: <letter>

Question: {question}
Options:
{options}"""

def build_prompt(q):
    if q["dataset"] == "gsm8k":
        return MATH_PROMPT.format(question=q["question"])
    options = "\n".join(f"{label}. {text}" for label, text in q["options"].items())
    return CSQA_PROMPT.format(question=q["question"], options=options)

print(build_prompt(questions[-1]))

## 5. Generate the traces

Loads the model and answers every question with **greedy decoding** (no randomness), so re-running gives the same answers. This takes a few minutes for 10 questions per dataset.

In [ ]:
model_id = MODELS[MODEL]
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.padding_side = "left"              # needed to answer several questions at once
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="auto")
model.eval()

# Llama's chat template inserts today's date; fix it so prompts are identical on every run.
template_args = {"date_string": "26 Jul 2024"} if MODEL.startswith("llama") else {}

def generate(prompts):
    chats = [
        tokenizer.apply_chat_template([{"role": "user", "content": p}], tokenize=False,
                                      add_generation_prompt=True, **template_args)
        for p in prompts
    ]
    inputs = tokenizer(chats, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, do_sample=False, max_new_tokens=MAX_NEW_TOKENS,
                                pad_token_id=tokenizer.pad_token_id)
    new_tokens = output[:, inputs["input_ids"].shape[1]:]          # keep only the answer
    return tokenizer.batch_decode(new_tokens, skip_special_tokens=True)

**Quick test:** one simple question, to check the model loaded and answers normally before the real run.

In [ ]:
print(generate(["Who are you?"])[0])

**Generate traces for all questions.**

In [ ]:
# Check that the earlier cells were run (sections 1, 3, 4 and 5)
for name in ["MODEL", "BATCH_SIZE", "questions", "build_prompt", "generate"]:
    if name not in globals():
        raise RuntimeError(f"'{name}' is missing: run the earlier cells first, or use Run All.")

traces = []
for start in range(0, len(questions), BATCH_SIZE):
    batch = questions[start:start + BATCH_SIZE]
    prompts = [build_prompt(q) for q in batch]
    for q, prompt, answer in zip(batch, prompts, generate(prompts)):
        traces.append({**q, "model": MODEL, "prompt": prompt, "raw_output": answer.strip()})
    print(f"{len(traces)}/{len(questions)} done")

## 6. Check the answers

For each trace:
- **Split into steps:** cut the text at every `Step k:`.
- **Read the final answer:** the last number after `Final answer:` (math), or the option letter (commonsense).
- **Compare with the correct answer.**

If there is no `Final answer:` line, the trace is marked `format_ok = False`. That is a formatting problem, not a reasoning error, and it is counted separately.

In [ ]:
STEP_RE = re.compile(r"^[\s*#>-]*step\s*(\d+)\s*[:.)]+\**\s*", re.IGNORECASE | re.MULTILINE)
FINAL_RE = re.compile(r"^[\s*#>-]*final\s+answer\s*\**\s*:\s*\**\s*(.*)$", re.IGNORECASE | re.MULTILINE)

def split_steps(text):
    markers = list(STEP_RE.finditer(text))
    return [text[m.end():(markers[i + 1].start() if i + 1 < len(markers) else len(text))].strip()
            for i, m in enumerate(markers)]

def read_final_answer(text, dataset):
    found = FINAL_RE.findall(text)
    if not found:
        return None
    answer = found[-1]
    if dataset == "gsm8k":
        numbers = re.findall(r"-?\d[\d,]*(?:\.\d+)?", answer.replace("$", ""))
        return numbers[-1].replace(",", "") if numbers else None
    letter = re.search(r"(?<![A-Za-z])([A-E])(?![A-Za-z])", answer)
    return letter.group(1) if letter else None

def is_correct(model_answer, gold, dataset):
    if model_answer is None:
        return False
    if dataset == "gsm8k":
        return abs(float(model_answer) - float(gold)) < 1e-6
    return model_answer == gold

for t in traces:
    body = FINAL_RE.split(t["raw_output"])[0]          # text before "Final answer:"
    t["steps"] = split_steps(body)
    t["model_answer"] = read_final_answer(t["raw_output"], t["dataset"])
    t["format_ok"] = t["model_answer"] is not None and len(t["steps"]) > 0
    t["correct"] = t["format_ok"] and is_correct(t["model_answer"], t["gold_answer"], t["dataset"])

print("checked", len(traces), "traces")

## 7. Look at the results

A small summary table, then a few traces to read. **Read several traces yourself:** this is the first look at how the model reasons and fails.

In [ ]:
df = pd.DataFrame(traces)
df["n_steps"] = df["steps"].map(len)
summary = df.groupby("dataset").agg(
    traces=("question_id", "size"),
    format_ok=("format_ok", "mean"),
    accuracy=("correct", "mean"),
    avg_steps=("n_steps", "mean"),
).round(2)
summary

In [ ]:
def show(t):
    print("=" * 80)
    print(f"[{t['dataset']}] {t['question_id']}   correct answer: {t['gold_answer']}   "
          f"model answer: {t['model_answer']}   {'CORRECT' if t['correct'] else 'WRONG'}")
    print("-" * 80)
    print(t["question"])
    if t["options"]:
        print(t["options"])
    print("-" * 80)
    for i, step in enumerate(t["steps"], start=1):
        print(f"[{i}] {step}")
    if not t["format_ok"]:
        print("FORMAT PROBLEM, raw output:\n" + t["raw_output"])

wrong = [t for t in traces if not t["correct"]]
print(f"{len(wrong)} wrong answers out of {len(traces)}\n")
for t in wrong[:5]:
    show(t)

## 8. Save

Saves one file per model in `/kaggle/working/`. Download it from the **Output** section of the right-hand panel (or after *Save Version*), so it can be added to the GitHub repo.

In [ ]:
out_path = f"/kaggle/working/traces_{MODEL}_{N_QUESTIONS}.jsonl"
with open(out_path, "w", encoding="utf-8") as f:
    for t in traces:
        f.write(json.dumps({**t, "max_new_tokens": MAX_NEW_TOKENS, "seed": SEED,
                            "transformers_version": transformers.__version__}, ensure_ascii=False) + "\n")
print("saved", len(traces), "traces to", out_path)